In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
import plotly.express as px
import pandas as pd

In [2]:
df = pd.read_csv('./data/USA_Housing.csv')

In [3]:
df.columns

Index(['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms',
       'Avg. Area Number of Bedrooms', 'Area Population', 'Price', 'Address'],
      dtype='object')

In [4]:
X = df.drop(['Address','Price'],axis=1)

In [5]:
from sklearn.preprocessing import MinMaxScaler

# Chuẩn hóa hai cột 'Area Income' và 'Area Population'
scaler = MinMaxScaler()
X[['Avg. Area Income', 'Area Population']] = scaler.fit_transform(X[['Avg. Area Income', 'Area Population']])


# Chia dữ liệu thành tập huấn luyện và tập kiểm thử
y = df['Price']  # Giả sử cột 'Price' là nhãn cần dự đoán
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=42)


In [6]:
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge


# Support Vector Regression
svr = SVR()
svr.fit(X_train, y_train)
svr_score = svr.score(X_test, y_test)
print(f"Support Vector Regression Score: {svr_score:.4f}")

# K-Nearest Neighbors Regression
knn = KNeighborsRegressor()
knn.fit(X_train, y_train)
knn_score = knn.score(X_test, y_test)
print(f"K-Nearest Neighbors Regression Score: {knn_score:.4f}")

# Random Forest Regression
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
rf_score = rf.score(X_test, y_test)
print(f"Random Forest Regression Score: {rf_score:.4f}")

# Decision Tree Regression
dt = DecisionTreeRegressor()
dt.fit(X_train, y_train)
dt_score = dt.score(X_test, y_test)
print(f"Decision Tree Regression Score: {dt_score:.4f}")

# Ridge Regression
ridge = Ridge()
ridge.fit(X_train, y_train)
ridge_score = ridge.score(X_test, y_test)
print(f"Ridge Regression Score: {ridge_score:.4f}")


Support Vector Regression Score: -0.0198
K-Nearest Neighbors Regression Score: 0.6243
Random Forest Regression Score: 0.8587
Decision Tree Regression Score: 0.6327
Ridge Regression Score: 0.8971


In [8]:
def tune_svr():
    param_grid = {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1, 0.2], 'kernel': ['linear', 'rbf']}
    grid = GridSearchCV(SVR(), param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best SVR Parameters: {grid.best_params_}")
    print(f"Best SVR Score: {grid.best_score_:.4f}\n")
    return grid

def tune_knn():
    param_grid = {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    grid = GridSearchCV(KNeighborsRegressor(), param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best KNN Parameters: {grid.best_params_}")
    print(f"Best KNN Score: {grid.best_score_:.4f}\n")
    return grid

def tune_rf():
    param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
    grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best Random Forest Parameters: {grid.best_params_}")
    print(f"Best Random Forest Score: {grid.best_score_:.4f}\n")
    return grid

def tune_dt():
    param_grid = {'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10]}
    grid = GridSearchCV(DecisionTreeRegressor(), param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best Decision Tree Parameters: {grid.best_params_}")
    print(f"Best Decision Tree Score: {grid.best_score_:.4f}\n")
    return grid

def tune_ridge():
    param_grid = {'alpha': [0.1, 1, 10, 100]}
    grid = GridSearchCV(Ridge(), param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Best Ridge Parameters: {grid.best_params_}")
    print(f"Best Ridge Score: {grid.best_score_:.4f}\n")
    return grid

# Chạy từng mô hình riêng biệt
svr_grid = tune_svr()
knn_grid = tune_knn()
rf_grid = tune_rf()
dt_grid = tune_dt()
ridge_grid = tune_ridge()


def visualize_results(grid, model_name):
    results_df = pd.DataFrame(grid.cv_results_)
    fig = px.parallel_coordinates(results_df, dimensions=[col for col in results_df.columns if 'param_' in col],
                                  color='mean_test_score',
                                  labels={col: col.replace('param_', '') for col in results_df.columns if 'param_' in col})
    fig.update_layout(title=f"Hyperparameter Tuning Results for {model_name}")
    fig.show()

visualize_results(svr_grid, 'SVR')
visualize_results(knn_grid, 'KNN')
visualize_results(rf_grid, 'Random Forest')
visualize_results(dt_grid, 'Decision Tree')
visualize_results(ridge_grid, 'Ridge')

Best SVR Parameters: {'C': 10, 'epsilon': 0.2, 'kernel': 'linear'}
Best SVR Score: 0.0491

Best KNN Parameters: {'n_neighbors': 5, 'weights': 'distance'}
Best KNN Score: 0.6223

Best Random Forest Parameters: {'max_depth': None, 'n_estimators': 200}
Best Random Forest Score: 0.8854

Best Decision Tree Parameters: {'max_depth': 10, 'min_samples_split': 10}
Best Decision Tree Score: 0.7838

Best Ridge Parameters: {'alpha': 0.1}
Best Ridge Score: 0.9179

